In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
tf.random.set_seed(42)
np.random.seed(42)

In [3]:
df = pd.read_csv("../data/asl_landmarks_final.csv")
X = df.drop(columns="label").values.reshape(-1, 21, 3)
le = LabelEncoder()
y = le.fit_transform(df["label"].values)
num_classes = len(le.classes_)

In [4]:
def normalize(lm):
    lm = lm - lm[:, 0:1, :]                     
    scale = np.linalg.norm(lm[:, 9:10, :2], axis=-1, keepdims=True)
    return lm / np.maximum(scale, 1e-6)

In [5]:
X = normalize(X).astype("float32")


In [6]:
X_tr, X_test, y_tr, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tr, y_tr, test_size=0.1765, stratify=y_tr, random_state=42)

In [7]:
cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(cw))

In [8]:
model = models.Sequential([
    layers.Input(shape=(21, 3)),
    layers.Conv1D(64, kernel_size=3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv1D(128, kernel_size=3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Flatten(),                      # konumsal bilgiyi koru
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

In [9]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 21, 64)              │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 21, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 21, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 21, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2688)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 2688)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         344,192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 28)                  │           3,612 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 373,916 (1.43 MB)

 Trainable params: 373,532 (1.42 MB)

 Non-trainable params: 384 (1.50 KB)

In [10]:
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=12,
                            restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
]

In [11]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    class_weight=class_weight,
    callbacks=cbs,
    verbose=2,
)

Epoch 1/150
49/49 - 11s - 221ms/step - accuracy: 0.6016 - loss: 1.4742 - val_accuracy: 0.6465 - val_loss: 2.7690 - learning_rate: 0.0010
Epoch 2/150
49/49 - 4s - 80ms/step - accuracy: 0.8533 - loss: 0.4993 - val_accuracy: 0.4230 - val_loss: 2.4748 - learning_rate: 0.0010
Epoch 3/150
49/49 - 2s - 49ms/step - accuracy: 0.9156 - loss: 0.3000 - val_accuracy: 0.4653 - val_loss: 2.1350 - learning_rate: 0.0010
Epoch 4/150
49/49 - 2s - 35ms/step - accuracy: 0.9208 - loss: 0.2509 - val_accuracy: 0.6344 - val_loss: 1.6047 - learning_rate: 0.0010
Epoch 5/150
49/49 - 3s - 55ms/step - accuracy: 0.9384 - loss: 0.1953 - val_accuracy: 0.7915 - val_loss: 1.2309 - learning_rate: 0.0010
Epoch 6/150
49/49 - 2s - 34ms/step - accuracy: 0.9422 - loss: 0.2018 - val_accuracy: 0.7855 - val_loss: 0.9187 - learning_rate: 0.0010
Epoch 7/150
49/49 - 2s - 31ms/step - accuracy: 0.9539 - loss: 0.1448 - val_accuracy: 0.8822 - val_loss: 0.5223 - learning_rate: 0.0010
Epoch 8/150
49/49 - 2s - 31ms/step - accuracy: 0.9591

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

Test accuracy: 0.9728


In [13]:
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           A      1.000     1.000     1.000        10
           B      1.000     1.000     1.000        11
           C      1.000     0.900     0.947        10
           D      1.000     1.000     1.000        13
           E      1.000     1.000     1.000        12
           F      1.000     1.000     1.000        15
           G      1.000     1.000     1.000        12
           H      1.000     1.000     1.000        12
           I      1.000     1.000     1.000        10
           J      1.000     1.000     1.000        19
           K      1.000     1.000     1.000        14
           L      1.000     1.000     1.000        12
           M      0.667     1.000     0.800         8
           N      1.000     0.333     0.500         6
           O      1.000     1.000     1.000        12
           P      1.000     1.000     1.000        11
           Q      1.000     0.900     0.947        10
           R      1.000    

In [14]:
model.save("../model/asl_gesture_cnn.keras")